# Movie Rating Prediction with Python

Predicting an IMDb movie's rating from **Genre**, **Director**, and **Actor** features, using historical Indian movie data.


## 1. Load the data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

RANDOM_STATE = 42

df = pd.read_csv("IMDb_Movies_India.csv", encoding="latin1")
print(df.shape)
df.head()


(15509, 10)


                                 Name    Year Duration            Genre  Rating Votes            Director       Actor 1             Actor 2          Actor 3
0                                         NaN      NaN            Drama     NaN   NaN       J.S. Randhawa      Manmauji              Birbal  Rajendra Bhatia
1  #Gadhvi (He thought he was Gandhi)  (2019)  109 min            Drama     7.0     8       Gaurav Bakshi  Rasika Dugal      Vivek Ghamande    Arvind Jangid
2                         #Homecoming  (2021)   90 min   Drama, Musical     NaN   NaN  Soumyajit Majumdar  Sayani Gupta   Plabita Borthakur       Roy Angana
3                             #Yaaram  (2019)  110 min  Comedy, Romance     4.4    35          Ovais Khan       Prateik          Ishita Raj  Siddhant Kapoor
4                   ...And Once Again  (2010)  105 min            Drama     NaN   NaN        Amol Palekar  Rajat Kapoor  Rituparna Sengupta      Antara Mali

## 2. Clean the columns

`Year`, `Duration`, and `Votes` are stored as text with extra characters (e.g. `(2019)`, `109 min`, `1,086`) and need to be converted to numbers. Rows without a `Rating` (the prediction target) are dropped, since they can't be used for training or evaluation.

In [2]:
df["Year"] = df["Year"].astype(str).str.extract(r"(\d{4})").astype(float)
df["Duration"] = df["Duration"].astype(str).str.extract(r"(\d+)").astype(float)
df["Votes"] = (
    df["Votes"].astype(str).str.replace(",", "", regex=False).str.extract(r"(\d+)").astype(float)
)

df = df.dropna(subset=["Rating"]).copy()
print(f"Shape after dropping missing Rating: {df.shape}")

df["Year"] = df["Year"].fillna(df["Year"].median())
df["Duration"] = df["Duration"].fillna(df["Duration"].median())
df["Votes"] = df["Votes"].fillna(0)
for col in ["Genre", "Director", "Actor 1", "Actor 2", "Actor 3"]:
    df[col] = df[col].fillna("Unknown")

df["Votes_log"] = np.log1p(df["Votes"])


Shape after dropping missing Rating: (7919, 10)


## 3. Feature engineering

- **Genre**: take the first-listed genre per movie, one-hot encode the top 15 genres (rest grouped as "Other").
- **Director / Actors**: these have thousands of unique names, so one-hot encoding isn't practical. Instead each name is **target-encoded** — replaced with the average rating of movies they've been in historically, smoothed toward the overall average so rare names don't get an unreliable estimate.

In [3]:
df["Genre_main"] = df["Genre"].apply(lambda g: g.split(",")[0].strip())
top_genres = df["Genre_main"].value_counts().nlargest(15).index
df["Genre_main"] = df["Genre_main"].where(df["Genre_main"].isin(top_genres), "Other")
genre_dummies = pd.get_dummies(df["Genre_main"], prefix="Genre")

global_mean = df["Rating"].mean()

def smoothed_target_encode(series, target, m=5):
    stats = target.groupby(series).agg(["mean", "count"])
    smoothed = (stats["mean"] * stats["count"] + global_mean * m) / (stats["count"] + m)
    return series.map(smoothed)

feature_cols = ["Year", "Duration", "Votes_log"]
X = pd.concat([df[feature_cols], genre_dummies], axis=1)
y = df["Rating"]


## 4. Train / test split

Director and actor encodings are recomputed using **only the training fold's** ratings, so no information from the test set leaks into the features (an easy mistake with target encoding).

In [4]:
X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, random_state=RANDOM_STATE
)

def encode_with_train_stats(train_series, train_target, test_series, m=5):
    stats = train_target.groupby(train_series).agg(["mean", "count"])
    smoothed = (stats["mean"] * stats["count"] + global_mean * m) / (stats["count"] + m)
    train_enc = train_series.map(smoothed)
    test_enc = test_series.map(smoothed).fillna(global_mean)
    return train_enc, test_enc

for col, enc_col in [("Director", "Director_enc"), ("Actor 1", "Actor1_enc"),
                      ("Actor 2", "Actor2_enc"), ("Actor 3", "Actor3_enc")]:
    tr_enc, te_enc = encode_with_train_stats(df_train[col], df_train["Rating"], df_test[col])
    X_train.loc[:, enc_col] = tr_enc.values
    X_test.loc[:, enc_col] = te_enc.values


## 5. Train and compare models

Three regressors are compared: a simple **Linear Regression** baseline, and two tree ensembles (**Random Forest**, **Gradient Boosting**) that can capture non-linear interactions between features.

In [5]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = []
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results.append({"Model": name, "R2": round(r2, 4), "MAE": round(mae, 4), "RMSE": round(rmse, 4)})
    fitted[name] = model

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
results_df


               Model      R2     MAE    RMSE
0  Linear Regression  0.2663  0.9079  1.1679
2  Gradient Boosting  0.2643  0.9006  1.1696
1      Random Forest  0.2354  0.9135  1.1923

## 6. Feature importance

Which features matter most to the best-performing model?

In [6]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted[best_name]
print(f"Best model: {best_name}")

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
    importances.sort_values(ascending=False).head(10)
else:
    coefs = pd.Series(best_model.coef_, index=X_train.columns)
    coefs.sort_values(key=abs, ascending=False).head(10)


Best model: Linear Regression


## 7. Try a prediction

Predict the rating for a hypothetical new movie by supplying its year, duration, vote count, genre, director, and lead actors.

In [7]:
def predict_rating(year, duration, votes, genre, director, actor1, actor2="Unknown", actor3="Unknown"):
    row = pd.DataFrame([{
        "Year": year, "Duration": duration, "Votes_log": np.log1p(votes),
    }])
    for g in genre_dummies.columns:
        row[g] = 1 if g == f"Genre_{genre}" else 0
    if f"Genre_{genre}" not in genre_dummies.columns:
        row["Genre_Other"] = 1

    dir_stats = df_train["Rating"].groupby(df_train["Director"]).agg(["mean", "count"])
    def lookup(name, stats_source_col):
        s = df_train[stats_source_col]
        stats = df_train["Rating"].groupby(s).agg(["mean", "count"])
        if name in stats.index:
            m, c = stats.loc[name]
            return (m * c + global_mean * 5) / (c + 5)
        return global_mean

    row["Director_enc"] = lookup(director, "Director")
    row["Actor1_enc"] = lookup(actor1, "Actor 1")
    row["Actor2_enc"] = lookup(actor2, "Actor 2")
    row["Actor3_enc"] = lookup(actor3, "Actor 3")

    row = row.reindex(columns=X_train.columns, fill_value=0)
    return best_model.predict(row)[0]

# Example usage:
predicted = predict_rating(
    year=2022, duration=130, votes=500,
    genre="Drama", director="Unknown", actor1="Unknown"
)
print(f"Predicted rating: {predicted:.2f}")


Predicted rating: 7.73
